# Notebook 2: Index Computation

## Purpose
Compute **theory-aligned indices** for all books & segments.

## Indices (as per hypotheses)
- `Love-over-Sex`: `(commitment_hea + tenderness_emotion) - explicit`
- `HEA Index`: `commitment_hea + symbolic_gifts + festive_rituals`
- `Luxury × Love`: `luxury × (commitment_hea + tenderness)`
- `Protective – Jealous`: `protectiveness - jealousy`
- `Dark-vs-Tender`: `(neg_affect + threat_dark) - tenderness`
- `Miscommunication Balance`: `(commitment + tenderness + repair) - miscommunication`
- Segment-wise: same indices per begin/middle/end

In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

## 1. Load Preprocessed Data

In [ ]:
PROJECT_ROOT = Path().resolve().parent.parent.parent.parent
INPUT_FILE = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis" / "book_category_props.csv"
OUTPUT_DIR = PROJECT_ROOT / "results" / "stage10_correlation_analysis" / "statistical_analysis"

# Load data from Notebook 1
df = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df)} rows")
print(f"Categories: {df['main_category_id'].nunique()}")
print(f"Books: {df['book_id'].nunique()}")

## 2. Pivot to Wide Format (One Row Per Book)

In [ ]:
# Pivot category proportions to wide format
# Each book becomes one row with columns for each category proportion

book_props_wide = df.pivot_table(
    index='book_id',
    columns='main_category_id',
    values='prop',
    fill_value=0.0
).reset_index()

# Merge back metadata
metadata_cols = ['book_id'] + [col for col in df.columns if col not in ['book_id', 'main_category_id', 'prop', 'n_sentences', 'total_sentences']]
metadata = df[metadata_cols].drop_duplicates(subset=['book_id'])
book_props_wide = book_props_wide.merge(metadata, on='book_id', how='left')

print(f"Wide format: {len(book_props_wide)} books, {book_props_wide.shape[1]} columns")

## 3. Compute Indices

In [ ]:
# TODO: Map category names to your actual taxonomy categories
# This is a template - adjust category names based on your taxonomy

def compute_indices(df_wide):
    """Compute all theory-aligned indices."""
    
    # Love-over-Sex Index
    # H1: Top > Trash
    df_wide['love_over_sex'] = (
        df_wide.get('commitment_hea', 0) + 
        df_wide.get('tenderness_emotion', 0) - 
        df_wide.get('explicit', 0)
    )
    
    # HEA Index
    # H2: Top > Trash
    df_wide['hea_index'] = (
        df_wide.get('commitment_hea', 0) + 
        df_wide.get('symbolic_gifts', 0) + 
        df_wide.get('festive_rituals', 0)
    )
    
    # Luxury × Love
    df_wide['luxury_x_love'] = (
        df_wide.get('luxury', 0) * 
        (df_wide.get('commitment_hea', 0) + df_wide.get('tenderness', 0))
    )
    
    # Protective – Jealous
    # H4: Top > Trash
    df_wide['protective_minus_jealous'] = (
        df_wide.get('protectiveness', 0) - 
        df_wide.get('jealousy', 0)
    )
    
    # Dark-vs-Tender
    # H5: Top < Trash
    df_wide['dark_vs_tender'] = (
        df_wide.get('neg_affect', 0) + 
        df_wide.get('threat_dark', 0) - 
        df_wide.get('tenderness', 0)
    )
    
    # Miscommunication Balance
    df_wide['miscommunication_balance'] = (
        df_wide.get('commitment', 0) + 
        df_wide.get('tenderness', 0) + 
        df_wide.get('repair', 0) - 
        df_wide.get('miscommunication', 0)
    )
    
    return df_wide

book_props_wide = compute_indices(book_props_wide)
print("✓ Computed all indices")

## 4. Compute Segment-Level Indices (if available)

In [ ]:
# TODO: If chapter_category_props.csv exists, compute indices per segment
# Same logic as above, but grouped by (book_id, segment)

## 5. Save Outputs

In [ ]:
# Save book-level indices
output_file = OUTPUT_DIR / "indices_book.csv"
book_props_wide.to_csv(output_file, index=False)
print(f"✓ Saved: {output_file}")

# TODO: Save chapter-level indices if computed
# chapter_indices_file = OUTPUT_DIR / "indices_chapter.csv"

## Summary

All theory-aligned indices computed. Next: Notebook 3 (Exploratory Analysis)